# Lab 05: Convolutional Neural Networks (CNNs) & Feature Map Visualizations

Welcome to Laboratory 05! In this lab, we transition to **Computer Vision** with **2D Convolutional Neural Networks (CNNs)**:
1. **2D Discrete Convolution Arithmetic**: Understand receptive fields, padding modes, strides, and spatial output dimensions.
2. **Convolutional Filters from First Principles**: Apply edge detection Sobel kernels to 2D tensors using `torch.nn.functional.conv2d`.
3. **Deep CNN Architecture (`CustomCNN`)**: Build a modular convolutional backbone with `Conv2d`, `BatchNorm2d`, `ReLU`, and `MaxPool2d` for CIFAR-10 classification.
4. **Intermediate Activation Feature Maps**: Extract and visualize the internal representations learned across convolutional layers.


## 1. Technical Preliminaries & Setup


In [ ]:
# Import core libraries for deep learning, computer vision, datasets, and visualization
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Seed random generators for experiment reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Select GPU acceleration if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Compute Device:', device)


## 2. 2D Convolution Arithmetic & Edge Detection Filters

### Mathematical Formulation: 2D Spatial Cross-Correlation
For an input tensor $\mathbf{X} \in \mathbb{R}^{H_{in} \times W_{in}}$ and a kernel filter $\mathbf{K} \in \mathbb{R}^{K_h \times K_w}$ with padding $P$ and stride $S$:
$$(\mathbf{X} \star \mathbf{K})(i, j) = \sum_{m=0}^{K_h - 1} \sum_{n=0}^{K_w - 1} \mathbf{X}(i \cdot S + m, j \cdot S + n) \mathbf{K}(m, n)$$

### Output Spatial Dimension Formula:
$$H_{out} = \left\lfloor \frac{H_{in} - K_h + 2P}{S} \right\rfloor + 1, \quad W_{out} = \left\lfloor \frac{W_{in} - K_w + 2P}{S} \right\rfloor + 1$$


### Custom Function: `apply_edge_filter`
The function below applies a vertical Sobel edge detector kernel to an input image tensor to highlight intensity transitions.
* **Input Shape**: `(1, 1, H, W)` single-channel batched image tensor.
* **Kernel Shape**: `(1, 1, 3, 3)` vertical Sobel derivative filter.
* **Output Shape**: `(1, 1, H-2, W-2)` when padding=0, stride=1.


In [ ]:
def apply_edge_filter(image_tensor: torch.Tensor) -> torch.Tensor:
    """Applies a vertical edge detection kernel to an input image tensor using F.conv2d.
    
    Args:
        image_tensor: 4D Tensor of shape (Batch=1, Channels=1, Height, Width)
    Returns:
        Filtered feature map highlighting vertical edge boundaries.
    """
    # Define vertical Sobel edge detection kernel of shape (Out_Channels=1, In_Channels=1, 3, 3)
    edge_kernel = torch.tensor([[[[ 1.0, 0.0, -1.0],
                                  [ 2.0, 0.0, -2.0],
                                  [ 1.0, 0.0, -1.0]]]], dtype=torch.float32)
    
    # Execute 2D convolution without padding (valid convolution) with stride 1
    feature_map = F.conv2d(image_tensor, edge_kernel, stride=1, padding=0)
    return feature_map

# Create an 8x8 synthetic image with a sharp vertical intensity step boundary (Left=1.0, Right=0.0)
synthetic_img = torch.ones((1, 1, 8, 8))
synthetic_img[:, :, :, 4:] = 0.0  # Zero out right half to create sharp vertical edge

# Apply vertical edge detection filter
edge_response = apply_edge_filter(synthetic_img)

print('Input Image Shape:       ', synthetic_img.shape)
print('Output Feature Map Shape:', edge_response.shape)
print('Filtered Response Values:\n', edge_response.squeeze().numpy())


## 3. Deep CNN Architecture for CIFAR-10 Classification

### Architecture Overview: `CustomCNN`
The `CustomCNN` model processes 3-channel color images through two hierarchical convolutional feature extraction blocks followed by a dense classification head:
* **Input**: Batched RGB images of shape `(Batch_Size, 3, 32, 32)`.
* **Block 1**:
  * `Conv2d(3, 32, kernel_size=3, padding=1)`: Preserves spatial resolution `(32x32)` while increasing depth to 32 channels.
  * `BatchNorm2d(32)`: Normalizes channel activations, reducing internal covariate shift.
  * `ReLU()`: Element-wise non-linearity.
  * `MaxPool2d(2, 2)`: Downsamples spatial resolution by factor of 2: `(32x32) -> (16x16)`.
* **Block 2**:
  * `Conv2d(32, 64, kernel_size=3, padding=1)`: Increases depth to 64 feature maps.
  * `BatchNorm2d(64)`: Normalization.
  * `ReLU()`: Non-linearity.
  * `MaxPool2d(2, 2)`: Downsamples spatial dimensions: `(16x16) -> (8x8)`.
* **Dense Classifier**:
  * `Flatten()`: Unrolls `(64, 8, 8)` tensor into 4,096-dimensional vector.
  * `Linear(4096, 128)` followed by `ReLU()` and `Linear(128, 10)` output logits.


In [ ]:
# Define the Custom CNN Architecture for CIFAR-10 Image Classification
class CustomCNN(nn.Module):
    """Two-stage Convolutional Neural Network with Batch Normalization for CIFAR-10."""
    def __init__(self, num_classes: int = 10):
        super(CustomCNN, self).__init__()
        
        # Stage 1: Low-level spatial feature extractor (3 -> 32 channels)
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1), # (B, 32, 32, 32)
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)                                # (B, 32, 16, 16)
        )
        
        # Stage 2: Mid-level compositional feature extractor (32 -> 64 channels)
        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1), # (B, 64, 16, 16)
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)                                 # (B, 64, 8, 8)
        )
        
        # Dense Classification Head
        self.classifier = nn.Sequential(
            nn.Flatten(),                                                        # Unroll to (B, 64*8*8)
            nn.Linear(in_features=64 * 8 * 8, out_features=128),                 # Fully connected layer
            nn.ReLU(),
            nn.Dropout(p=0.25),                                                  # Prevent dense overfitting
            nn.Linear(in_features=128, out_features=num_classes)                 # 10 output class logits
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Hierarchical forward pass through feature stages and classifier
        feat1 = self.block1(x)
        feat2 = self.block2(feat1)
        logits = self.classifier(feat2)
        return logits

# Data Preprocessing: Normalize CIFAR-10 with per-channel RGB statistics
cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load datasets
train_set = datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform)
test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)

# Initialize model, loss criterion, and Adam optimizer
cnn_model = CustomCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

print('CustomCNN Architecture Model Summary:\n', cnn_model)


### CNN Training Loop on CIFAR-10
The cell below executes mini-batch training over multiple epochs, tracking cross-entropy loss.


In [ ]:
# Train the Custom CNN model
num_epochs = 3
for epoch in range(num_epochs):
    cnn_model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = cnn_model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass & parameter updates
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100.0
    print(f'Epoch [{epoch+1}/{num_epochs}] -> Training Loss: {epoch_loss:.4f} | Training Accuracy: {epoch_acc:.2f}%')


## 4. Summary & Takeaways
1. **Spatial Invariance**: Convolutions exploit local spatial correlations using shared weight kernels across the entire image.
2. **Batch Normalization**: Stabilizes intermediate distributions, enabling faster convergence and higher learning rates.
3. **Hierarchical Representations**: Early CNN layers capture primitive edges/textures, while deeper layers compose high-level semantics.
